# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amah67/mlintern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)



## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking and Scoring would be the task type.

Why - Editiors do not have time to rewrite every lots of pages. Instead of a result that is yes/no, it would be more beneficial if they had a ranked queue so they know what pages to fix first depending on the performance.



In [7]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Total articles to prioritize: {len(df):,}")
print(
    f"Articles losing traffic (trend_pct < 0): {(df['trend_pct'] < 0).sum():,}"
)

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.
Total articles to prioritize: 30,000
Articles losing traffic (trend_pct < 0): 19,715


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target - Search traffic drop (trend_pct < 0 or rank drop in avg_position

Label source - This is information measured from actual Google performance data in a later time window.

In [8]:
# Verify target label is observed in the data
print("Target Variable Summary (trend_pct):")
print(df["trend_pct"].describe())

# Define observed binary decay indicator for validation
df["is_decaying"] = (df["trend_pct"] < 0).astype(int)
print("\nObserved Decay Rate:")
print(df["is_decaying"].value_counts(normalize=True))

Target Variable Summary (trend_pct):
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Observed Decay Rate:
is_decaying
1    0.657167
0    0.342833
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success Metric - Precision@K

Why - If an editor chooses the 20 pages marked out by our model, then 75 percent of those (15 pages out of the 20) must  have experienced search traffic decline.

In [9]:
# Baseline metric check: Precision@20 of a naive rule (e.g., lowest CTR)
k = 20
naive_top_k = df.sort_values(by="ctr", ascending=True).head(k)
naive_precision_at_k = naive_top_k["is_decaying"].mean()

print(f"Naive Rule Baseline Precision@{k}: {naive_precision_at_k:.2%}")

Naive Rule Baseline Precision@20: 55.00%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = One specific content page (content_id) for a given client (client_id).

In [10]:
# Display unit of analysis and verify unique grain
print(f"Dataset Shape: {df.shape}")

# Check grain uniqueness
duplicate_check = df.duplicated(subset=["content_id"]).sum()
print(f"Duplicate content_ids: {duplicate_check} (Must be 0)")

# Show the real dataframe slice
df[
    [
        "content_id",
        "client_id",
        "search_volume",
        "avg_position",
        "ctr",
        "trend_pct",
    ]
].head(5)

Dataset Shape: (30000, 45)
Duplicate content_ids: 0 (Must be 0)


,content_id,client_id,search_volume,avg_position,ctr,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,10.6,0.76,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,20.3,0.05,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,36.5,0.09,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,6.2,0.49,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,44.0,0.13,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple, static rule such as “if avg_position > 10” does not work because there are many interacting variables that influence search visibility like search volume, competition level, keyword intent, and ai_traffic_pct exposure. High volume page being two positions lower is much more important than the page with low volume being 10 positions down. Machine Learning can handle all of these interactions easily.